In [ ]:
import os

os.environ["JAVA_HOME"] = (
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
)

In [ ]:
# Configuration Spark optimisée pour MacBook Pro M3 Max (36GB, 14 cœurs)
# Rappels : réserver ~8-10GB pour macOS, utiliser tous les cœurs disponibles
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ObRail_AERO_Analysis") \
    .master("local[14]") \
    .config("spark.driver.memory", "30g") \
    .config("spark.driver.maxResultSize", "6g") \
    .config("spark.sql.shuffle.partitions", "56") \
    .config("spark.default.parallelism", "56") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.memory.fraction", "0.75") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.sql.autoBroadcastJoinThreshold", "20971520") \
    .config("spark.local.dir", "/tmp/spark-temp") \
    .config("spark.ui.showConsoleProgress", "false") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

# Configuration du niveau de log
spark.sparkContext.setLogLevel("WARN")

print(f"✓ Spark Session créée avec succès")
print(f"  - Version Spark : {spark.version}")
print(f"  - Master : {spark.sparkContext.master}")
print(f"  - Mémoire Driver : 26 GB")
print(f"  - Cœurs utilisés : 14")
print(f"  - Partitions shuffle : 56")
print(f"  - Application ID : {spark.sparkContext.applicationId}")

In [ ]:
df_train_trip = spark.read.parquet("../data/processed/train_trips.parquet")
df_flight_trip = spark.read.parquet("../data/processed/flight.parquet")
df_localite = spark.read.parquet("../data/processed/localite.parquet")
df_emissions = spark.read.parquet("../data/processed/emission.parquet")
df_trip_match = spark.read.parquet("../data/processed/stop_matching.parquet")

print(f"\n✓ Données chargées :")
print(f"  - Trip : {df_train_trip.count()} lignes")
print(f"  - Fly Trip : {df_flight_trip.count()} lignes")
print(f"  - Localité : {df_localite.count()} lignes")
print(f"  - Emissions : {df_emissions.count()} lignes")
print(f"  - Trip Match : {df_trip_match.count()} lignes")

print(f"\n✓ Aperçu des données :")
print(f"\nTrip :")
df_train_trip.show(5, truncate=False)
print(f"\nFly Trip :")
df_flight_trip.show(5, truncate=False)
print(f"\nLocalité :")
df_localite.show(5, truncate=False)
print(f"\nEmissions :")
df_emissions.show(5, truncate=False)
print(f"\nTrip Match :")
df_trip_match.show(5, truncate=False)

print(f"\n✓ Schéma des données :")
print(f"\nTrip :")
df_train_trip.printSchema()
print(f"\nFly Trip :")
df_flight_trip.printSchema()
print(f"\nLocalité :")
df_localite.printSchema()
print(f"\nEmissions :")
df_emissions.printSchema()
print(f"\nTrip Match :")
df_trip_match.printSchema()

In [ ]:
# ═══════════════════════════════════════════════════════════
# GOLD TRAIN — Passage silver étoile -> O/D aggloméré
# ═══════════════════════════════════════════════════════════
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1) Enrichissement localités + facteur d'émission
#    - city_name depuis df_localite
#    - co2_per_pkm depuis df_emissions (gCO2e / p.km)
df_train_enriched = (
    df_train_trip.alias("t")
    .join(
        df_localite.select(
            F.col("city_id").alias("loc_city_id"),
            F.col("city_name")
        ).alias("l"),
        F.col("t.city_id") == F.col("l.loc_city_id"),
        "left"
    )
    .join(
        df_emissions.select(
            F.col("emission_ref").cast("long").alias("em_ref"),
            F.col("emission_gCO2e_per_p_km").alias("co2_per_pkm")
        ).alias("e"),
        F.col("t.emission_ref").cast("long") == F.col("e.em_ref"),
        "left"
    )
    .drop("loc_city_id", "em_ref")
)

# 2) Distance cumulée depuis le 1er arrêt du trip
#    IMPORTANT: un trip est unique par (source, trip_id)
w_cumul = (
    Window.partitionBy("source", "trip_id")
    .orderBy("stop_sequence")
    .rowsBetween(Window.unboundedPreceding, -1)
)

df_work_train = (
    df_train_enriched
    .withColumn(
        "cumul_dist_m",
        F.coalesce(
            F.sum(F.coalesce(F.col("segment_dist_m"), F.lit(0.0))).over(w_cumul),
            F.lit(0.0)
        )
    )
    .repartition(200, "source", "trip_id")
    .persist()
)

print(f"Stops train total : {df_work_train.count():,}")

# 3) Côté départ
df_dep_train = df_work_train.select(
    F.col("source").alias("trip_source"),
    F.col("trip_id"),
    F.col("stop_sequence").alias("dep_seq"),
    F.col("stop_name").alias("departure_station"),
    F.col("city_name").alias("departure_city"),
    F.col("country_name").alias("departure_country"),
    F.col("departure_time"),
    F.col("parent_station").alias("departure_parent_station"),
    F.col("cumul_dist_m").alias("dep_cumul"),
)

# 4) Côté arrivée
df_arr_train = df_work_train.select(
    F.col("source").alias("arr_source"),
    F.col("trip_id").alias("_trip_id_arr"),
    F.col("stop_sequence").alias("arr_seq"),
    F.col("stop_name").alias("arrival_station"),
    F.col("city_name").alias("arrival_city"),
    F.col("country_name").alias("arrival_country"),
    F.col("arrival_time"),
    F.col("parent_station").alias("arrival_parent_station"),
    F.col("cumul_dist_m").alias("arr_cumul"),
)

# 5) Attributs trip-level (1 ligne par trip)
df_trip_attrs_train = (
    df_work_train
    .select(
        "source", "trip_id", "route_id", "service_id", "route_type",
        "route_short_name", "route_long_name", "trip_short_name",
        "agency_name", "agency_timezone",
        "start_date", "end_date", "days_of_week", "is_night_train", "co2_per_pkm"
    )
    .dropDuplicates(["source", "trip_id"])
)

# 6) Génération de toutes les paires O/D (dep_seq < arr_seq)
df_pairs_train = (
    df_dep_train.join(
        df_arr_train,
        (F.col("trip_source") == F.col("arr_source")) &
        (F.col("trip_id") == F.col("_trip_id_arr")) &
        (F.col("dep_seq") < F.col("arr_seq")),
        "inner"
    )
    .withColumn("distance_m", F.round(F.col("arr_cumul") - F.col("dep_cumul"), 2))
    .withColumn("distance_km", F.round(F.col("distance_m") / F.lit(1000.0), 2))
    .drop(
        "arr_source", "_trip_id_arr", "dep_cumul", "arr_cumul",
        "dep_seq", "arr_seq", "distance_m"
    )
    .withColumnRenamed("trip_source", "source")
)

# 7) Dataset gold train (avant dédoublonnage O/D)
df_gold_train_raw = (
    df_pairs_train
    .join(df_trip_attrs_train, ["source", "trip_id"], "inner")
    .withColumn("mode", F.lit("train"))
    .withColumn("destination", F.coalesce(F.col("route_long_name"), F.col("trip_short_name")))
    .withColumn("route_type", F.col("route_type").cast("string"))
    .withColumnRenamed("start_date", "service_start_date")
    .withColumnRenamed("end_date", "service_end_date")
    .withColumn(
        "emissions_co2",
        F.round(F.col("distance_km") * F.col("co2_per_pkm"), 3)
    )
)

# 8) Dédoublonnage par gare+ville+horaires de départ/arrivée
#    Fusion days_of_week par OR bit à bit, ex: 1100110 + 1010101 -> 1110111
dedup_keys = [
    "source", "mode",
    "departure_station", "departure_city", "departure_country", "departure_parent_station", "departure_time",
    "arrival_station", "arrival_city", "arrival_country", "arrival_parent_station", "arrival_time",
    "route_type"
]

def merged_days_expr(col_name="days_of_week"):
    return F.concat(
        *[
            F.max(
                F.when(
                    F.substring(F.coalesce(F.col(col_name), F.lit("0000000")), i, 1).isin("0", "1"),
                    F.substring(F.coalesce(F.col(col_name), F.lit("0000000")), i, 1).cast("int")
                ).otherwise(F.lit(0))
            ).cast("string")
            for i in range(1, 8)
        ]
    )

preserve_cols = [
    "trip_id", "destination", "trip_short_name",
    "agency_name", "agency_timezone",
    "service_id", "route_id", "route_short_name", "route_long_name",
    "service_start_date", "service_end_date",
    "is_night_train", "distance_km", "co2_per_pkm", "emissions_co2"
]

agg_exprs = [merged_days_expr("days_of_week").alias("days_of_week")]
agg_exprs.extend([F.first(F.col(c), ignorenulls=True).alias(c) for c in preserve_cols])

final_cols_train = [
    "source", "trip_id", "mode", "destination", "trip_short_name",
    "agency_name", "agency_timezone",
    "service_id", "route_id", "route_type", "route_short_name", "route_long_name",
    "departure_station", "departure_city", "departure_country", "departure_time",
    "departure_parent_station",
    "arrival_station", "arrival_city", "arrival_country", "arrival_time",
    "arrival_parent_station",
    "service_start_date", "service_end_date", "days_of_week",
    "is_night_train", "distance_km", "co2_per_pkm", "emissions_co2"
]

df_gold_train = (
    df_gold_train_raw
    .groupBy(*dedup_keys)
    .agg(*agg_exprs)
    .select(final_cols_train)
)

print(f"Lignes gold train (avant dédoublonnage) : {df_gold_train_raw.count():,}")
print(f"Lignes gold train (après dédoublonnage) : {df_gold_train.count():,}")

df_work_train.unpersist()

In [ ]:
# ═══════════════════════════════════════════════════════════
# GOLD FLIGHT + UNION GOLD AGGLOMÉRÉ
# ═══════════════════════════════════════════════════════════
from pyspark.sql.types import StringType

# 1) Préparation localités origine/destination
df_loc_origin = (
    df_localite
    .select(
        F.col("city_id").alias("origin_city_id_l"),
        F.col("city_name").alias("departure_city")
    )
)

df_loc_dest = (
    df_localite
    .select(
        F.col("city_id").alias("dest_city_id_l"),
        F.col("city_name").alias("arrival_city")
    )
)

# 2) Mapping route_long_name depuis trajet_type
#    format attendu: <small|medium|large>_<small|medium|large>_<1000km|3000km|...>
size_label = F.create_map(
    F.lit("small"), F.lit("Petit Aéroport"),
    F.lit("medium"), F.lit("Moyen Aéroport"),
    F.lit("large"), F.lit("Grand Aéroport")
)

haul_label = (
    F.when(F.col("haul_band") == F.lit("1000km"), F.lit("Court courrier"))
     .when(F.col("haul_band") == F.lit("3000km"), F.lit("Moyen Courrier"))
     .otherwise(F.lit("Long Courrier"))
)

# 3) Construction gold flight
df_gold_flight = (
    df_flight_trip.alias("f")
    .join(
        df_loc_origin.alias("lo"),
        F.col("f.origin_city_id") == F.col("lo.origin_city_id_l"),
        "left"
    )
    .join(
        df_loc_dest.alias("ld"),
        F.col("f.dest_city_id") == F.col("ld.dest_city_id_l"),
        "left"
    )
    .join(
        df_emissions.select(
            F.col("emission_ref").cast("long").alias("em_ref"),
            F.col("emission_gCO2e_per_p_km").alias("co2_per_pkm")
        ).alias("e"),
        F.col("f.emission_ref").cast("long") == F.col("e.em_ref"),
        "left"
    )
    .withColumn("source", F.lit("ourairports"))
    .withColumn("trip_id", F.concat_ws("__", F.col("f.origin_id").cast("string"), F.col("f.dest_id").cast("string")))
    .withColumn("mode", F.lit("flight"))
    .withColumn("destination", F.concat_ws(" - ", F.col("f.origin_name"), F.col("f.dest_name")))
    .withColumn("trip_short_name", F.concat_ws(" - ", F.col("f.origin_ident"), F.col("f.dest_ident")))
    .withColumn("route_type", F.col("f.trajet_type"))
    .withColumn("route_tokens", F.split(F.col("f.trajet_type"), "_"))
    .withColumn("origin_size", F.element_at(F.col("route_tokens"), 1))
    .withColumn("dest_size", F.element_at(F.col("route_tokens"), 2))
    .withColumn("haul_band", F.element_at(F.col("route_tokens"), 3))
    .withColumn(
        "route_long_name",
        F.concat_ws(
            " - ",
            haul_label,
            F.concat(
                F.element_at(size_label, F.col("origin_size")),
                F.lit(" vers "),
                F.element_at(size_label, F.col("dest_size"))
            )
        )
    )
    .withColumn("distance_km", F.round(F.col("f.distance_km"), 2))
    .withColumn("emissions_co2", F.round(F.col("distance_km") * F.col("co2_per_pkm"), 3))
    .select(
        "source",
        "trip_id",
        "mode",
        "destination",
        "trip_short_name",
        F.lit(None).cast(StringType()).alias("agency_name"),
        F.lit(None).cast(StringType()).alias("agency_timezone"),
        F.lit(None).cast(StringType()).alias("service_id"),
        F.lit(None).cast(StringType()).alias("route_id"),
        "route_type",
        F.lit(None).cast(StringType()).alias("route_short_name"),
        "route_long_name",
        F.col("f.origin_name").alias("departure_station"),
        "departure_city",
        F.col("f.origin_country").alias("departure_country"),
        F.lit(None).cast(StringType()).alias("departure_time"),
        F.lit(None).cast(StringType()).alias("departure_parent_station"),
        F.col("f.dest_name").alias("arrival_station"),
        "arrival_city",
        F.col("f.dest_country").alias("arrival_country"),
        F.lit(None).cast(StringType()).alias("arrival_time"),
        F.lit(None).cast(StringType()).alias("arrival_parent_station"),
        F.lit(None).cast(StringType()).alias("service_start_date"),
        F.lit(None).cast(StringType()).alias("service_end_date"),
        F.lit(None).cast(StringType()).alias("days_of_week"),
        F.lit(None).cast("boolean").alias("is_night_train"),
        "distance_km",
        "co2_per_pkm",
        "emissions_co2"
    )
)

# 4) Harmonisation schéma flight (train déjà harmonisé dans cellule 4)
final_cols = [
    "source", "trip_id", "mode", "destination", "trip_short_name",
    "agency_name", "agency_timezone",
    "service_id", "route_id", "route_type", "route_short_name", "route_long_name",
    "departure_station", "departure_city", "departure_country", "departure_time",
    "departure_parent_station",
    "arrival_station", "arrival_city", "arrival_country", "arrival_time",
    "arrival_parent_station",
    "service_start_date", "service_end_date", "days_of_week",
    "is_night_train", "distance_km", "co2_per_pkm", "emissions_co2"
]

df_gold_flight = df_gold_flight.select(final_cols)

# 5) Union gold aggloméré
#    (train + avion dans un dataset unique)
df_gold_agg = df_gold_train.unionByName(df_gold_flight)

print(f"Lignes gold train   : {df_gold_train.count():,}")
print(f"Lignes gold flight  : {df_gold_flight.count():,}")
print(f"Lignes gold total   : {df_gold_agg.count():,}")

df_gold_agg.printSchema()
df_gold_agg.show(10, truncate=False)


print("✓ Dataset gold aggloméré écrit: ../data/processed/gold_routes_agglomere.parquet")

In [ ]:
# ═══════════════════════════════════════════════════════════
# GOLD COMPARAISON TRAIN VS AVION (via stop_matching)
# ═══════════════════════════════════════════════════════════
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1) Utilitaires durée
sec_in_day = F.lit(24 * 3600)

def hms_to_seconds(col_name: str):
    parts = F.split(F.col(col_name), ":")
    return (
        F.coalesce(parts.getItem(0).cast("int"), F.lit(0)) * F.lit(3600)
        + F.coalesce(parts.getItem(1).cast("int"), F.lit(0)) * F.lit(60)
        + F.coalesce(parts.getItem(2).cast("int"), F.lit(0))
    )

# 2) Candidats TRAIN (0 correspondance)
dep_sec = hms_to_seconds("departure_time")
arr_sec = hms_to_seconds("arrival_time")
train_duration_min = F.when(
    F.col("departure_time").isNotNull() & F.col("arrival_time").isNotNull(),
    F.round(
        F.when(arr_sec >= dep_sec, arr_sec - dep_sec)
         .otherwise((arr_sec + sec_in_day) - dep_sec) / F.lit(60.0),
        1,
    ),
).otherwise(F.round(F.col("distance_km") / F.lit(90.0) * F.lit(60.0), 1))

df_cmp_train = (
    df_gold_train
    .select(
        "source", "trip_id", "departure_station", "departure_city", "departure_country",
        "arrival_station", "arrival_city", "arrival_country",
        "departure_parent_station", "arrival_parent_station",
        "departure_time", "arrival_time",
        "distance_km", "emissions_co2", "days_of_week", "mode"
    )
    .withColumn("candidate_mode", F.lit("train"))
    .withColumn("candidate_id", F.col("trip_id"))
    .withColumn("correspondence_count", F.lit(0))
    .withColumn("duration_min", train_duration_min)
    .withColumn("comparison_distance_km", F.col("distance_km"))
    .withColumn("comparison_emissions_co2", F.col("emissions_co2"))
)

# 3) Meilleur vol par paire d'aéroports (durée puis émissions)
df_flight_options = (
    df_flight_trip.alias("f")
    .join(
        df_emissions.select(
            F.col("emission_ref").cast("long").alias("em_ref"),
            F.col("emission_gCO2e_per_p_km").alias("co2_per_pkm")
        ).alias("e"),
        F.col("f.emission_ref").cast("long") == F.col("e.em_ref"),
        "left"
    )
    .select(
        F.col("f.origin_ident").alias("dep_airport_id"),
        F.col("f.dest_ident").alias("arr_airport_id"),
        F.col("f.origin_name").alias("flight_departure_station"),
        F.col("f.dest_name").alias("flight_arrival_station"),
        F.round(F.col("f.distance_km"), 2).alias("flight_distance_km"),
        F.round(F.col("f.distance_km") / F.lit(700.0) * F.lit(60.0) + F.lit(75.0), 1).alias("flight_duration_min"),
        F.round(F.col("f.distance_km") * F.col("co2_per_pkm"), 3).alias("flight_emissions_co2")
    )
)

w_flight_pair = (
    Window.partitionBy("dep_airport_id", "arr_airport_id")
    .orderBy(F.col("flight_duration_min").asc_nulls_last(), F.col("flight_emissions_co2").asc_nulls_last())
)

df_flight_best_pair = (
    df_flight_options
    .withColumn("rn", F.row_number().over(w_flight_pair))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 4) Mapping train OD -> aéroports proches (correspondances autorisées)
df_match_dep = df_trip_match.select(
    F.col("gare_id").alias("dep_gare_id"),
    F.col("airport_id").alias("dep_airport_id"),
    F.col("distance_km").alias("dep_access_km")
)

df_match_arr = df_trip_match.select(
    F.col("gare_id").alias("arr_gare_id"),
    F.col("airport_id").alias("arr_airport_id"),
    F.col("distance_km").alias("arr_access_km")
)

df_cmp_flight_from_train_od = (
    df_cmp_train.alias("t")
    .join(
        df_match_dep.alias("md"),
        F.col("t.departure_parent_station") == F.col("md.dep_gare_id"),
        "inner"
    )
    .join(
        df_match_arr.alias("ma"),
        F.col("t.arrival_parent_station") == F.col("ma.arr_gare_id"),
        "inner"
    )
    .join(
        df_flight_best_pair.alias("fb"),
        (F.col("md.dep_airport_id") == F.col("fb.dep_airport_id")) &
        (F.col("ma.arr_airport_id") == F.col("fb.arr_airport_id")),
        "inner"
    )
    .withColumn("candidate_mode", F.lit("flight"))
    .withColumn(
        "candidate_id",
        F.concat_ws("__", F.col("fb.dep_airport_id"), F.col("fb.arr_airport_id"))
    )
    .withColumn("access_km", F.round(F.col("md.dep_access_km") + F.col("ma.arr_access_km"), 2))
    .withColumn("correspondence_count", F.lit(2))
    .withColumn("duration_min", F.round(F.col("fb.flight_duration_min") + (F.col("access_km") / F.lit(40.0) * F.lit(60.0)), 1))
    .withColumn("comparison_distance_km", F.round(F.col("fb.flight_distance_km") + F.col("access_km"), 2))
    .withColumn("comparison_emissions_co2", F.col("fb.flight_emissions_co2"))
    .select(
        "t.source", "t.trip_id", "t.departure_station", "t.departure_city", "t.departure_country",
        "t.arrival_station", "t.arrival_city", "t.arrival_country",
        "t.departure_parent_station", "t.arrival_parent_station",
        "t.days_of_week",
        "candidate_mode", "candidate_id", "correspondence_count",
        "duration_min", "comparison_distance_km", "comparison_emissions_co2"
    )
)

# 5) Union candidats + sélection du meilleur trajet par O/D
#    Priorité: durée la plus courte -> moins de correspondances -> moins d'émissions
df_cmp_train_norm = (
    df_cmp_train
    .select(
        "source", "trip_id", "departure_station", "departure_city", "departure_country",
        "arrival_station", "arrival_city", "arrival_country",
        "departure_parent_station", "arrival_parent_station",
        "days_of_week",
        "candidate_mode", "candidate_id", "correspondence_count",
        "duration_min", "comparison_distance_km", "comparison_emissions_co2"
    )
)

df_gold_compare_candidates = df_cmp_train_norm.unionByName(df_cmp_flight_from_train_od)

w_best = (
    Window.partitionBy(
        "departure_station", "departure_city", "departure_country",
        "arrival_station", "arrival_city", "arrival_country"
    )
    .orderBy(
        F.col("duration_min").asc_nulls_last(),
        F.col("correspondence_count").asc(),
        F.col("comparison_emissions_co2").asc_nulls_last()
    )
)

df_gold_compare_best = (
    df_gold_compare_candidates
    .withColumn("selection_rank", F.row_number().over(w_best))
    .withColumn("is_best_option", F.col("selection_rank") == 1)
)

print(f"Candidats comparaison : {df_gold_compare_candidates.count():,}")
print(f"Trajets O/D évalués   : {df_gold_compare_best.select('departure_station', 'arrival_station').distinct().count():,}")
print(f"Meilleures options    : {df_gold_compare_best.filter(F.col('is_best_option')).count():,}")

# 6) Écriture
(
    df_gold_compare_candidates
    .repartition(200)
    .write
    .mode("overwrite")
    .parquet("../data/processed/gold_train_flight_compare_candidates.parquet")
)

(
    df_gold_compare_best
    .filter(F.col("is_best_option"))
    .drop("selection_rank", "is_best_option")
    .repartition(200)
    .write
    .mode("overwrite")
    .parquet("../data/processed/gold_train_flight_compare_best.parquet")
)

print("✓ gold_train_flight_compare_candidates.parquet écrit")
print("✓ gold_train_flight_compare_best.parquet écrit")